In [0]:
%sql
-- Read each observed response file as one record, preserving its original JSON.
CREATE OR REPLACE TEMP VIEW places_bronze_source AS
SELECT
    path AS source_file_path,
    modificationTime AS source_modified_at,
    length AS source_size_bytes,
    decode(content, 'UTF-8') AS raw_json
FROM read_files(
    '/Volumes/places_ratings/landing/source_files/observed',
    format => 'binaryFile',
    recursiveFileLookup => true,
    pathGlobFilter => '*.json'
);

In [0]:
%sql
-- Measure source file counts and total size before writing the Bronze table.
SELECT
    COUNT(*) AS source_files,
    COUNT(DISTINCT source_file_path) AS distinct_paths,
    SUM(source_size_bytes) AS total_bytes,
    ROUND(
        SUM(source_size_bytes) / (1024.0 * 1024.0),
        2
    ) AS total_mib
FROM places_bronze_source;

In [0]:
%sql
select * from places_bronze_source

In [0]:
%sql
-- Extract source metadata while retaining the complete original response.
CREATE OR REPLACE TEMP VIEW places_bronze_prepared AS
SELECT
    source_file_path,
    source_modified_at,
    source_size_bytes,
    sha2(raw_json, 256) AS source_content_hash,
    get_json_object(raw_json, '$.extraction_id') AS extraction_id,
    get_json_object(raw_json, '$.data_kind') AS data_kind,
    get_json_object(raw_json, '$.study_area') AS study_area,
    get_json_object(raw_json, '$.requested_category') AS requested_category,
    COALESCE(
        get_json_object(raw_json, '$.extraction.mapping_version'),
        'legacy'
    ) AS mapping_version,
    TRY_CAST(
        get_json_object(raw_json, '$.extraction.extracted_at_utc')
        AS TIMESTAMP
    ) AS extracted_at_utc,
    raw_json
FROM places_bronze_source;

In [0]:
%sql
-- Validate extracted metadata before persisting the Bronze table.
SELECT
    mapping_version,
    COUNT(*) AS source_files,
    COUNT(DISTINCT extraction_id) AS distinct_extraction_ids,
    COUNT_IF(extraction_id IS NULL) AS missing_extraction_id,
    COUNT_IF(study_area IS NULL) AS missing_study_area,
    COUNT_IF(requested_category IS NULL) AS missing_category,
    COUNT_IF(extracted_at_utc IS NULL) AS missing_or_invalid_timestamp,
    COUNT_IF(data_kind IS NULL OR data_kind <> 'observed')
        AS unexpected_data_kind
FROM places_bronze_prepared
GROUP BY mapping_version
ORDER BY mapping_version;

In [0]:
%sql
-- Create the Bronze Delta table with one row per source file.
CREATE TABLE IF NOT EXISTS places_ratings.bronze.api_responses (
    source_file_path STRING,
    source_modified_at TIMESTAMP,
    source_size_bytes BIGINT,
    source_content_hash STRING,
    extraction_id STRING,
    data_kind STRING,
    study_area STRING,
    requested_category STRING,
    mapping_version STRING,
    extracted_at_utc TIMESTAMP,
    raw_json STRING,
    bronze_ingested_at TIMESTAMP
)
USING DELTA
COMMENT 'Original Places API response files with extraction and ingestion metadata';

In [0]:
%sql
-- Insert previously unseen source files without duplicating existing records.
MERGE INTO places_ratings.bronze.api_responses AS target
USING places_bronze_prepared AS source
ON target.source_file_path = source.source_file_path

WHEN NOT MATCHED THEN
INSERT (
    source_file_path,
    source_modified_at,
    source_size_bytes,
    source_content_hash,
    extraction_id,
    data_kind,
    study_area,
    requested_category,
    mapping_version,
    extracted_at_utc,
    raw_json,
    bronze_ingested_at
)
VALUES (
    source.source_file_path,
    source.source_modified_at,
    source.source_size_bytes,
    source.source_content_hash,
    source.extraction_id,
    source.data_kind,
    source.study_area,
    source.requested_category,
    source.mapping_version,
    source.extracted_at_utc,
    source.raw_json,
    current_timestamp()
);

In [0]:
%sql
-- Verify Bronze row counts and uniqueness after ingestion.
SELECT
    mapping_version,
    COUNT(*) AS bronze_rows,
    COUNT(DISTINCT source_file_path) AS distinct_source_files,
    COUNT(DISTINCT extraction_id) AS distinct_extraction_ids
FROM places_ratings.bronze.api_responses
GROUP BY mapping_version
ORDER BY mapping_version;

In [0]:
%sql
select * from places_ratings.bronze.api_responses